# 3DFin data preparation

This notebook prepares the 3DFin tree-detection and measurement datasets used in the dissertation analysis.

The original 3DFin output spreadsheets are kept unchanged. Final GT-to-3DFin matches, tree-identity confidence and manual tree-height attribution decisions come from the manually checked terrain workbooks.

The notebook creates:

- the high-confidence measurement dataset before manual TH-attribution QC
- the final measurement dataset after manual TH-attribution QC
- the candidate-level tree-detection classifications
- the final tree-detection metrics

Earlier pre- and post-QC output files are used only to check that the reconstructed workflow reproduces the previous results. They are not used to decide which observations are retained.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd


# Find the main dissertation-analysis project folder
# The project root is identified by checking for both the 'data' and 'notebooks' folders
def find_project_root(start_path):
    """
    Find the main dissertation-analysis folder.
    """

    start_path = Path(start_path).resolve()

    for folder in [start_path, *start_path.parents]:
        if (
            (folder / "data").is_dir()
            and (folder / "notebooks").is_dir()
        ):
            return folder

    raise FileNotFoundError(
        "Could not find the project root. "
        "Expected a folder containing both 'data' and 'notebooks'."
    )


# Find the project root from the notebook's current location
PROJECT_DIR = find_project_root(Path.cwd())

DATA_DIR = PROJECT_DIR / "data"


# Define the main folders used in the 3DFin workflow
RAW_3DFIN_DIR = DATA_DIR / "raw" / "3dfin"
MANUAL_3DFIN_DIR = DATA_DIR / "manual_qc" / "3dfin"
VALIDATION_3DFIN_DIR = DATA_DIR / "validation" / "3dfin"
PROCESSED_3DFIN_DIR = DATA_DIR / "processed" / "3dfin"


# Create the processed-data folder if it does not already exist
PROCESSED_3DFIN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print(f"Project root found: {PROJECT_DIR.name}")

Project root found: dissertation-analysis


In [2]:
# Inspect the 3DFin source folders
# List the files available for each terrain before defining the inputs
# This makes it clear which source, manual-QC and validation files are present in the project

def list_files(folder):
    """List the files stored directly within a project folder."""

    print(f"\n{folder.relative_to(PROJECT_DIR)}")

    if not folder.exists():
        print("Folder not found")
        return

    files = sorted(
        path.name
        for path in folder.iterdir()
        if path.is_file()
    )

    if not files:
        print("Folder is empty")
        return

    for filename in files:
        print(f"  {filename}")


# Check the raw, manual-QC and validation folders used in the 3DFin workflow
for terrain_folder in [
    RAW_3DFIN_DIR / "easy",
    RAW_3DFIN_DIR / "intermediate",
    RAW_3DFIN_DIR / "challenging",
    MANUAL_3DFIN_DIR / "easy",
    MANUAL_3DFIN_DIR / "intermediate",
    MANUAL_3DFIN_DIR / "challenging",
    VALIDATION_3DFIN_DIR,
]:
    list_files(terrain_folder)


data/raw/3dfin/easy
  20_20591171_n_e_dvojskener_orezane.xlsx
  20_20591171_n_e_hovermap_orezane.xlsx
  20_20591171_n_e_jednoskenerovy_orezane.xlsx

data/raw/3dfin/intermediate
  10_20611471_int_dvojskener_orezane.xlsx
  10_20611471_int_hovermap_orezane.xlsx
  10_20611471_n_int_jednoskener_orezane.xlsx

data/raw/3dfin/challenging
  15_20531271_n_ch_dvojskener_orezane.xlsx
  15_20531271_n_ch_hovermap_orezane.xlsx
  15_20531271_n_ch_jednoskener_orezane.xlsx

data/manual_qc/3dfin/easy
  Final_Easy_Terrain.xlsx

data/manual_qc/3dfin/intermediate
  Final_Intermediate_Terrain.xlsx

data/manual_qc/3dfin/challenging
  Final_Challenging_Terrain.xlsx

data/validation/3dfin
  3dfin_postqc_per_tree.csv
  3dfin_postqc_summary.csv
  3dfin_prethqc_per_tree.csv
  3dfin_prethqc_summary.csv


In [3]:
# Helper used for data checks throughout the notebook

def require(condition, message):
    """Stop the notebook if an expected data check fails."""

    if not condition:
        raise AssertionError(message)

In [4]:
# Identify the 3DFin source files
# Each terrain contains one original 3DFin Excel output for each scanner
# The files are identified using scanner-specific text in the filename
# The original source files remain unchanged

SCANNER_FILE_TOKENS = {
    "Single sensor": [
        "jednoskener",
        "jednoskenner",
    ],
    "Dual sensor": [
        "dvojskener",
        "dvojskenner",
    ],
    "Hovermap": [
        "hovermap",
    ],
}


def find_scanner_file(folder, scanner):
    """
    Find the Excel file for one scanner within a terrain folder.
    """

    tokens = SCANNER_FILE_TOKENS[scanner]

    # Search for Excel files containing one of the expected scanner-name variants while ignoring temporary Excel files
    matches = [
        path
        for path in folder.glob("*.xlsx")
        if not path.name.startswith("~$")
        and any(
            token in path.name.lower()
            for token in tokens
        )
    ]

    # There should be exactly one source file for each scanner within each terrain folder
    require(
        len(matches) == 1,
        (
            f"Expected exactly one {scanner} file in "
            f"{folder.relative_to(PROJECT_DIR)}, "
            f"but found {len(matches)}."
        ),
    )

    return matches[0]


# Find the raw 3DFin source file for every terrain-scanner combination
RAW_3DFIN_FILES = {}

for terrain in ["Easy", "Intermediate", "Challenging"]:

    terrain_folder = (
        RAW_3DFIN_DIR / terrain.lower()
    )

    for scanner in [
        "Single sensor",
        "Dual sensor",
        "Hovermap",
    ]:

        RAW_3DFIN_FILES[(terrain, scanner)] = (
            find_scanner_file(
                terrain_folder,
                scanner,
            )
        )


# The manually checked terrain workbooks have fixed final filenames

MANUAL_3DFIN_FILES = {
    "Easy":
        MANUAL_3DFIN_DIR
        / "easy"
        / "Final_Easy_Terrain.xlsx",

    "Intermediate":
        MANUAL_3DFIN_DIR
        / "intermediate"
        / "Final_Intermediate_Terrain.xlsx",

    "Challenging":
        MANUAL_3DFIN_DIR
        / "challenging"
        / "Final_Challenging_Terrain.xlsx",
}


# Check that all three manual-QC workbooks are available
for terrain, path in MANUAL_3DFIN_FILES.items():

    require(
        path.exists(),
        f"Missing manual-QC workbook: {path.name}",
    )


# Print the source files used by the notebook as a final check
print("Raw 3DFin files:\n")

for (terrain, scanner), path in RAW_3DFIN_FILES.items():
    print(
        f"{terrain:13s} | "
        f"{scanner:13s} | "
        f"{path.name}"
    )


print("\nManual-QC files:\n")

for terrain, path in MANUAL_3DFIN_FILES.items():
    print(
        f"{terrain:13s} | {path.name}"
    )

Raw 3DFin files:

Easy          | Single sensor | 20_20591171_n_e_jednoskenerovy_orezane.xlsx
Easy          | Dual sensor   | 20_20591171_n_e_dvojskener_orezane.xlsx
Easy          | Hovermap      | 20_20591171_n_e_hovermap_orezane.xlsx
Intermediate  | Single sensor | 10_20611471_n_int_jednoskener_orezane.xlsx
Intermediate  | Dual sensor   | 10_20611471_int_dvojskener_orezane.xlsx
Intermediate  | Hovermap      | 10_20611471_int_hovermap_orezane.xlsx
Challenging   | Single sensor | 15_20531271_n_ch_jednoskener_orezane.xlsx
Challenging   | Dual sensor   | 15_20531271_n_ch_dvojskener_orezane.xlsx
Challenging   | Hovermap      | 15_20531271_n_ch_hovermap_orezane.xlsx

Manual-QC files:

Easy          | Final_Easy_Terrain.xlsx
Intermediate  | Final_Intermediate_Terrain.xlsx
Challenging   | Final_Challenging_Terrain.xlsx


In [5]:
# Confirm the earlier 3DFin validation outputs
# These files are used only to check that the cleaned workflow reproduces the earlier 3DFin results
# They are not used to decide which trees are retained in the analysis

VALIDATION_3DFIN_FILES = {
    "pre_per_tree":
        VALIDATION_3DFIN_DIR
        / "3dfin_prethqc_per_tree.csv",

    "pre_summary":
        VALIDATION_3DFIN_DIR
        / "3dfin_prethqc_summary.csv",

    "post_per_tree":
        VALIDATION_3DFIN_DIR
        / "3dfin_postqc_per_tree.csv",

    "post_summary":
        VALIDATION_3DFIN_DIR
        / "3dfin_postqc_summary.csv",
}


# Check that all expected validation files are available
for name, path in VALIDATION_3DFIN_FILES.items():

    require(
        path.exists(),
        f"Missing validation file: {path.name}",
    )


print(
    "All 9 raw files, 3 manual-QC workbooks, "
    "and 4 validation files were found."
)

All 9 raw files, 3 manual-QC workbooks, and 4 validation files were found.


In [6]:
# Inspect the raw and manual-QC spreadsheet structures
# The original 3DFin exports and the manually checked terrain workbooks use different layouts
# so their rows and columns are inspected before any analysis tables are created
# openpyxl is used because the source files are in .xlsx format

print("RAW 3DFin file structure:\n")

for (terrain, scanner), path in RAW_3DFIN_FILES.items():

    # The raw 3DFin exports contain two header rows before the main table so they are skipped when the file is loaded
    df = pd.read_excel(
        path,
        skiprows=2,
        engine="openpyxl",
    )

    print(f"{terrain} | {scanner}")
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print()


print("\nMANUAL-QC workbook structure:\n")

for terrain, path in MANUAL_3DFIN_FILES.items():

    df = pd.read_excel(
        path,
        engine="openpyxl",
    )

    print(terrain)
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print()

RAW 3DFin file structure:

Easy | Single sensor
Rows: 9
Columns: ['Unnamed: 0', 'Unnamed: 1', 'TH', 'DBH', 'X', 'Y']

Easy | Dual sensor
Rows: 9
Columns: ['Unnamed: 0', 'Unnamed: 1', 'TH', 'DBH', 'X', 'Y']

Easy | Hovermap
Rows: 9
Columns: ['Unnamed: 0', 'Unnamed: 1', 'TH', 'DBH', 'X', 'Y']

Intermediate | Single sensor
Rows: 23
Columns: ['Unnamed: 0', 'Unnamed: 1', 'TH', 'DBH', 'X', 'Y']

Intermediate | Dual sensor
Rows: 24
Columns: ['Unnamed: 0', 'Unnamed: 1', 'TH', 'DBH', 'X', 'Y']

Intermediate | Hovermap
Rows: 25
Columns: ['Unnamed: 0', 'Unnamed: 1', 'TH', 'DBH', 'X', 'Y']

Challenging | Single sensor
Rows: 27
Columns: ['Unnamed: 0', 'Unnamed: 1', 'TH', 'DBH', 'X', 'Y']

Challenging | Dual sensor
Rows: 30
Columns: ['Unnamed: 0', 'Unnamed: 1', 'TH', 'DBH', 'X', 'Y']

Challenging | Hovermap
Rows: 30
Columns: ['Unnamed: 0', 'Unnamed: 1', 'TH', 'DBH', 'X', 'Y']


MANUAL-QC workbook structure:

Easy
Rows: 9
Columns: ['GT', 'GT TH', 'GT DBH', 'J', 'J TH', 'J TH Issue', 'J DBH', 'D', 'D 

In [7]:
# Standardise the 3DFin Tree IDs
# Tree IDs appear in several formats, including T1, 1 and 1.0
# Convert these to one consistent integer identifier

def clean_tree_id(value):
    """Convert different Tree-ID formats to an integer."""

    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "na",
        "n/a",
        "none",
        "-"
    }:
        return np.nan

    # Extract the numeric part of the Tree ID
    numbers = re.findall(r"\d+", text)

    if not numbers:
        return np.nan

    return int(numbers[0])


# Read one original 3DFin tree table

def read_3dfin_tree_table(path):
    """
    Read one original 3DFin output spreadsheet.
    The spreadsheets contain two header rows before the main
    tree table. The Tree-ID column is stored immediately before
    the TH column.
    The original source file is not modified.
    """

    df = pd.read_excel(
        path,
        skiprows=2,
        engine="openpyxl",
    )

    # Remove any extra spaces from the Excel column names
    df.columns = [
        str(column).strip()
        for column in df.columns
    ]


    # Check that the main 3DFin measurement columns are present
    required_columns = [
        "TH",
        "DBH",
        "X",
        "Y",
    ]

    for column in required_columns:
        require(
            column in df.columns,
            (
                f"{path.name}: expected column "
                f"'{column}' was not found."
            ),
        )


    # In the original 3DFin exports, the Tree-ID column sits directly before the TH column

    th_position = df.columns.get_loc("TH")

    require(
        th_position > 0,
        f"{path.name}: could not identify the Tree-ID column.",
    )

    tree_id_column = df.columns[th_position - 1]


    # Keep only the tree ID and measurements needed later
    out = df[
        [
            tree_id_column,
            "TH",
            "DBH",
            "X",
            "Y",
        ]
    ].copy()


    out.columns = [
        "Tree_ID_raw",
        "TH",
        "DBH",
        "X",
        "Y",
    ]


    # Convert the original Tree-ID values to a consistent numeric identifier
    out["Tree_ID"] = (
        out["Tree_ID_raw"]
        .map(clean_tree_id)
    )


    # Remove blank or footer rows that do not represent trees
    out = out[
        out["Tree_ID"].notna()
    ].copy()


    out["Tree_ID"] = (
        out["Tree_ID"]
        .astype(int)
    )


    # Convert the measurement columns to numeric values
    for column in [
        "TH",
        "DBH",
        "X",
        "Y",
    ]:
        out[column] = pd.to_numeric(
            out[column],
            errors="coerce",
        )


    # Each Tree ID should appear only once within a scanner dataset
    # Stop if duplicate IDs are found 
    require(
        not out["Tree_ID"].duplicated().any(),
        f"{path.name}: duplicate 3DFin Tree IDs were found.",
    )


    return out.reset_index(drop=True)


# Load all nine original 3DFin candidate tables

raw_3dfin_tables = []


for (terrain, scanner), path in RAW_3DFIN_FILES.items():

    df = read_3dfin_tree_table(path)

    # Add the terrain, scanner and source filename so each candidate can be traced back to its original dataset
    df["Terrain"] = terrain
    df["Scanner"] = scanner
    df["source_file"] = path.name

    raw_3dfin_tables.append(df)

    print(
        f"{terrain:13s} | "
        f"{scanner:13s} | "
        f"{len(df):2d} candidates"
    )


# Combine all nine scanner-terrain tables into one candidate table
raw_3dfin_candidates = pd.concat(
    raw_3dfin_tables,
    ignore_index=True,
)


# Check the expected total number of original 3DFin candidates
require(
    len(raw_3dfin_candidates) == 186,
    (
        "Expected 186 original 3DFin candidates across "
        "the nine scanner-terrain datasets."
    ),
)


# Each Tree ID should be unique within its terrain and scanner
require(
    not raw_3dfin_candidates.duplicated(
        subset=["Terrain", "Scanner", "Tree_ID"]
    ).any(),
    "Duplicate terrain + scanner + Tree ID combinations found.",
)


print(
    "\nTotal original 3DFin candidates:",
    len(raw_3dfin_candidates)
)

Easy          | Single sensor |  9 candidates
Easy          | Dual sensor   |  9 candidates
Easy          | Hovermap      |  9 candidates
Intermediate  | Single sensor | 23 candidates
Intermediate  | Dual sensor   | 24 candidates
Intermediate  | Hovermap      | 25 candidates
Challenging   | Single sensor | 27 candidates
Challenging   | Dual sensor   | 30 candidates
Challenging   | Hovermap      | 30 candidates

Total original 3DFin candidates: 186


In [8]:
# Load the three manually checked 3DFin workbooks
# Only small structural changes are made at this stage so that the three terrain workbooks can be handled consistently later
# No trees are filtered and no measurement values are changed

manual_3dfin = {}


for terrain, path in MANUAL_3DFIN_FILES.items():

    df = pd.read_excel(
        path,
        engine="openpyxl",
    )

    # Remove accidental leading or trailing spaces from the Excel column names
    df.columns = df.columns.str.strip()

    # The Easy workbook uses "ID CONF", while the Intermediate and Challenging workbooks use "CONF" for the same field.
    if "ID CONF" in df.columns:
        df = df.rename(
            columns={
                "ID CONF": "CONF"
            }
        )

    manual_3dfin[terrain] = df

    print(
        f"{terrain:13s} | "
        f"{len(df):2d} rows"
    )

Easy          |  9 rows
Intermediate  | 25 rows
Challenging   | 27 rows


In [9]:
# Inspect the manual-QC values before applying any filtering
# Check the values used for tree-identity confidence and manual TH validation before standardising them
# The GT DBH ranges are also checked because the Easy workbook stores GT DBH in a different unit from the other terrains

for terrain, df in manual_3dfin.items():

    print("\n" + "=" * 60)
    print(terrain.upper())
    print("=" * 60)

    print("\nTree-identity confidence values:")
    print(
        df["CONF"]
        .value_counts(dropna=False)
    )

    print("\nManual TH-validation values:")
    print(
        df["Manually Validated Sample for TH"]
        .value_counts(dropna=False)
    )

    print("\nGT DBH range:")
    print(
        "min =",
        pd.to_numeric(
            df["GT DBH"],
            errors="coerce"
        ).min(),
        "| max =",
        pd.to_numeric(
            df["GT DBH"],
            errors="coerce"
        ).max(),
    )


EASY

Tree-identity confidence values:
high    9
Name: CONF, dtype: int64

Manual TH-validation values:
NO     5
YES    4
Name: Manually Validated Sample for TH, dtype: int64

GT DBH range:
min = 248 | max = 631

INTERMEDIATE

Tree-identity confidence values:
HIGH    22
LOW      3
Name: CONF, dtype: int64

Manual TH-validation values:
NO     13
YES    12
Name: Manually Validated Sample for TH, dtype: int64

GT DBH range:
min = 0.18 | max = 0.365

CHALLENGING

Tree-identity confidence values:
high    19
low      8
Name: CONF, dtype: int64

Manual TH-validation values:
YES    15
NO     12
Name: Manually Validated Sample for TH, dtype: int64

GT DBH range:
min = 0.074 | max = 0.436


In [10]:
# Standardise the manual-QC workbooks
# Clean the formatting and measurement units so the three terrain workbooks can be handled consistently
# No decision is made here about which trees are retained for the analysis
# Confidence labels are standardised, manual TH decisions are converted to booleans, measurement fields are made numeric
# Easy GT DBH is converted from millimetres to metres.

MEASUREMENT_COLUMNS = [
    "GT TH",
    "GT DBH",
    "J TH",
    "J DBH",
    "D TH",
    "D DBH",
    "H TH",
    "H DBH",
]


manual_3dfin_clean = {}


for terrain, source_df in manual_3dfin.items():

    df = source_df.copy()

    # Standardise tree-identity confidence labels so they can be compared consistently across the three terrain workbooks
    df["CONF"] = (
        df["CONF"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Convert the manual TH-validation decisions from YES/NO labels to boolean values
    df["TH_validated"] = (
        df["Manually Validated Sample for TH"]
        .astype(str)
        .str.strip()
        .str.upper()
        .map({
            "YES": True,
            "NO": False,
        })
    )

    require(
        df["TH_validated"].notna().all(),
        (
            f"{terrain}: an unrecognised manual "
            "TH-validation value was found."
        ),
    )

    # Convert the GT and scanner measurement fields to numeric values before they are used in the analysis
    for column in MEASUREMENT_COLUMNS:

        require(
            column in df.columns,
            f"{terrain}: missing expected column '{column}'.",
        )

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    # Easy GT DBH was recorded in millimetres, while the scanner DBH estimates are already in metres
    if terrain == "Easy":
        df["GT DBH"] = df["GT DBH"] / 1000.0

    manual_3dfin_clean[terrain] = df


print("Manual-QC tables standardised.")

Manual-QC tables standardised.


In [11]:
# Check that the cleaned confidence labels, TH-validation decisions and GT DBH values now have the expected format

for terrain, df in manual_3dfin_clean.items():

    print("\n" + "=" * 60)
    print(terrain.upper())
    print("=" * 60)

    print(
        "Confidence values:",
        df["CONF"].value_counts(dropna=False).to_dict()
    )

    print(
        "TH-validation decisions:",
        df["TH_validated"].value_counts(dropna=False).to_dict()
    )

    print(
        "GT DBH range:",
        round(df["GT DBH"].min(), 3),
        "to",
        round(df["GT DBH"].max(), 3),
        "m",
    )


EASY
Confidence values: {'HIGH': 9}
TH-validation decisions: {False: 5, True: 4}
GT DBH range: 0.248 to 0.631 m

INTERMEDIATE
Confidence values: {'HIGH': 22, 'LOW': 3}
TH-validation decisions: {False: 13, True: 12}
GT DBH range: 0.18 to 0.365 m

CHALLENGING
Confidence values: {'HIGH': 19, 'LOW': 8}
TH-validation decisions: {True: 15, False: 12}
GT DBH range: 0.074 to 0.436 m


In [12]:
# Inspect the GT-to-3DFin Tree-ID assignments
# Check the identifiers exactly as they appear in the manually reviewed workbooks before converting them to numeric Tree IDs
# This helps preserve any unusual or ambiguous entries that may need to be handled separately

for terrain, df in manual_3dfin_clean.items():

    print("\n" + "=" * 70)
    print(terrain.upper())
    print("=" * 70)

    print(
        df[
            [
                "GT",
                "J",
                "D",
                "H",
                "CONF",
                "TH_validated",
            ]
        ].to_string(index=False)
    )


EASY
 GT  J  D  H  CONF  TH_validated
  6  4  4  4  HIGH          True
 13  7  7  8  HIGH         False
 17  6  6  6  HIGH         False
 20  8  8  7  HIGH         False
 21  9  9  9  HIGH          True
 27  5  5  5  HIGH          True
 31  2  2  2  HIGH         False
 33  3  3  3  HIGH          True
 34  1  1  1  HIGH         False

INTERMEDIATE
   GT     J     D     H  CONF  TH_validated
  1.0   9.0   8.0   7.0  HIGH          True
  2.0   3.0   3.0   3.0  HIGH          True
  4.0   5.0   5.0   5.0  HIGH         False
  5.0   2.0   1.0   1.0  HIGH         False
  8.0   8.0   9.0   9.0  HIGH         False
  9.0  11.0  12.0  11.0  HIGH          True
 11.0  10.0  10.0  10.0  HIGH          True
 14.0  16.0  18.0  16.0  HIGH         False
 15.0  18.0  17.0  17.0  HIGH         False
 16.0  22.0  23.0  22.0  HIGH         False
 17.0  19.0  19.0  19.0  HIGH          True
 19.0  21.0  22.0  21.0  HIGH         False
 20.0  23.0  24.0  23.0  HIGH          True
 22.0  15.0  16.0  15.0  HIGH     

In [13]:
# Convert the manual GT matching tables to long format
# Each row represents one GT tree for one scanner
# Tree-identity confidence and manual TH validation are kept as separate fields 
# because they describe different parts of the QC process

SCANNER_COLUMNS = {
    "Single sensor": {
        "tree_id": "J",
        "TH": "J TH",
        "DBH": "J DBH",
    },
    "Dual sensor": {
        "tree_id": "D",
        "TH": "D TH",
        "DBH": "D DBH",
    },
    "Hovermap": {
        "tree_id": "H",
        "TH": "H TH",
        "DBH": "H DBH",
    },
}


GT_TREE_COUNTS = {
    "Easy": 9,
    "Intermediate": 22,
    "Challenging": 27,
}


gt_match_rows = []


for terrain, df in manual_3dfin_clean.items():

    # Rows without a GT ID represent additional scanner candidates rather than GT-to-scanner matches
    gt_rows = df[
        df["GT"].notna()
    ].copy()

    require(
        len(gt_rows) == GT_TREE_COUNTS[terrain],
        (
            f"{terrain}: expected "
            f"{GT_TREE_COUNTS[terrain]} GT trees, "
            f"but found {len(gt_rows)}."
        ),
    )

    for _, row in gt_rows.iterrows():

        gt_id = clean_tree_id(row["GT"])

        for scanner, columns in SCANNER_COLUMNS.items():

            tree_id = clean_tree_id(
                row[columns["tree_id"]]
            )

            gt_match_rows.append({
                "Terrain": terrain,
                "Scanner": scanner,
                "GT_ID": gt_id,
                "Tree_ID": tree_id,
                "GT_TH": row["GT TH"],
                "GT_DBH": row["GT DBH"],
                "TH": row[columns["TH"]],
                "DBH": row[columns["DBH"]],
                "ID_confidence": row["CONF"],
                "TH_validated": row["TH_validated"],
            })


gt_matches_3dfin = pd.DataFrame(
    gt_match_rows
)


# Tree IDs can be missing when a GT tree was not matched to an individual 3DFin candidate
gt_matches_3dfin["Tree_ID"] = (
    gt_matches_3dfin["Tree_ID"]
    .astype("Int64")
)


# There are 58 GT trees in total and three scanner datasets giving 174 GT × scanner records
require(
    len(gt_matches_3dfin) == 174,
    "Expected 174 GT × scanner records.",
)


print(
    "GT-to-3DFin matching table:",
    len(gt_matches_3dfin),
    "rows"
)

print("\nGT trees by terrain:")

print(
    gt_matches_3dfin
    .groupby("Terrain")["GT_ID"]
    .nunique()
)

GT-to-3DFin matching table: 174 rows

GT trees by terrain:
Terrain
Challenging     27
Easy             9
Intermediate    22
Name: GT_ID, dtype: int64


In [14]:
# Preserve scanner candidates recorded without a GT match
# Some rows in the manually checked workbooks represent trees detected by a scanner but not present in the GT data
# Keep these as genuine unreferenced candidates rather than treating the missing GT value as missing data

unreferenced_rows = []


for terrain, df in manual_3dfin_clean.items():

    extra_rows = df[
        df["GT"].isna()
    ].copy()

    for _, row in extra_rows.iterrows():

        for scanner, columns in SCANNER_COLUMNS.items():

            tree_id = clean_tree_id(
                row[columns["tree_id"]]
            )

            if pd.isna(tree_id):
                continue

            unreferenced_rows.append({
                "Terrain": terrain,
                "Scanner": scanner,
                "Tree_ID": int(tree_id),
                "source": "manual_QC_workbook",
            })


manual_unreferenced_3dfin = pd.DataFrame(
    unreferenced_rows
)


print(
    "Scanner candidates explicitly recorded without a GT match:",
    len(manual_unreferenced_3dfin)
)

print()

print(
    manual_unreferenced_3dfin
    .to_string(index=False)
)

Scanner candidates explicitly recorded without a GT match: 5

      Terrain        Scanner  Tree_ID              source
 Intermediate  Single sensor        6  manual_QC_workbook
 Intermediate    Dual sensor        6  manual_QC_workbook
 Intermediate       Hovermap        6  manual_QC_workbook
 Intermediate       Hovermap       24  manual_QC_workbook
 Intermediate    Dual sensor       14  manual_QC_workbook


In [15]:
# Identify Tree IDs assigned to more than one GT tree
# If the same 3DFin Tree ID is linked to multiple GT trees within one scanner dataset,
# those GT trees were not recovered asseparate individual detections
# These cases are identified directly from the final manual matching table, not entered manually

require(
    len(gt_matches_3dfin) == 174,
    "Expected 174 GT × scanner records before detection QC.",
)


shared_tree_ids = (
    gt_matches_3dfin[
        gt_matches_3dfin["Tree_ID"].notna()
    ]
    .groupby(
        [
            "Terrain",
            "Scanner",
            "Tree_ID",
        ]
    )
    .agg(
        n_GT=("GT_ID", "nunique"),
        represented_GT_IDs=(
            "GT_ID",
            lambda values: ";".join(
                str(int(value))
                for value in sorted(values.unique())
            )
        ),
    )
    .reset_index()
)


# Keep only candidates linked to more than one GT tree
shared_tree_ids = (
    shared_tree_ids[
        shared_tree_ids["n_GT"] > 1
    ]
    .sort_values(
        [
            "Terrain",
            "Scanner",
            "Tree_ID",
        ]
    )
    .reset_index(drop=True)
)


print(
    "3DFin candidates assigned to more than one GT tree:",
    len(shared_tree_ids),
)

print()

print(
    shared_tree_ids.to_string(index=False)
)

3DFin candidates assigned to more than one GT tree: 8

     Terrain        Scanner  Tree_ID  n_GT represented_GT_IDs
 Challenging    Dual sensor        1     2              35;36
 Challenging    Dual sensor       18     2               7;32
 Challenging    Dual sensor       19     2              23;24
 Challenging       Hovermap        3     2              35;36
 Challenging       Hovermap       18     2               7;32
 Challenging       Hovermap       19     2              23;24
 Challenging  Single sensor        3     2              35;36
 Challenging  Single sensor       17     2              23;24


In [16]:
# Identify GT trees with no assigned 3DFin candidate
# These records represent GT trees that were not matched to an individual 3DFin detection for a given scanner

missing_assignments = (
    gt_matches_3dfin[
        gt_matches_3dfin["Tree_ID"].isna()
    ][
        [
            "Terrain",
            "Scanner",
            "GT_ID",
            "ID_confidence",
        ]
    ]
    .sort_values(
        [
            "Terrain",
            "Scanner",
            "GT_ID",
        ]
    )
    .reset_index(drop=True)
)


print(
    "GT × scanner records with no assigned 3DFin candidate:",
    len(missing_assignments),
)

print()

print(
    missing_assignments.to_string(index=False)
)

GT × scanner records with no assigned 3DFin candidate: 2

     Terrain        Scanner  GT_ID ID_confidence
 Challenging  Single sensor      7           LOW
 Challenging  Single sensor     32           LOW


In [17]:
# Build the final GT-level 3DFin detection table
# Detection is assessed separately from measurement accuracy
# A GT tree receives individual detection credit when it is matched to its own 3DFin candidate
# Missing matches receive no credit, and candidates shared by multiple GT trees are treated as under-segmented
# because the trees were not recovered as separate instances
# Tree-identity confidence is kept separate from detection
# For example, GT33 and GT34 have uncertain exact identities, but two distinct candidates are present
# so both still count as detections

gt_detection_3dfin = gt_matches_3dfin[
    [
        "Terrain",
        "Scanner",
        "GT_ID",
        "Tree_ID",
        "ID_confidence",
    ]
].copy()


# Use a temporary text key for the merge while preserving the original Tree_ID column
gt_detection_3dfin["_Tree_ID_key"] = (
    gt_detection_3dfin["Tree_ID"]
    .apply(
        lambda value:
        "MISSING"
        if pd.isna(value)
        else str(int(value))
    )
)


shared_for_merge = shared_tree_ids.copy()

shared_for_merge["_Tree_ID_key"] = (
    shared_for_merge["Tree_ID"]
    .apply(lambda value: str(int(value)))
)


gt_detection_3dfin = (
    gt_detection_3dfin
    .merge(
        shared_for_merge[
            [
                "Terrain",
                "Scanner",
                "_Tree_ID_key",
                "n_GT",
                "represented_GT_IDs",
            ]
        ],
        on=[
            "Terrain",
            "Scanner",
            "_Tree_ID_key",
        ],
        how="left",
    )
)


# Remove the temporary merge key once the shared-tree information has been added

gt_detection_3dfin = (
    gt_detection_3dfin
    .drop(columns="_Tree_ID_key")
)


# Candidates not identified as shared are assumed to represent one GT tree

gt_detection_3dfin["n_GT"] = (
    gt_detection_3dfin["n_GT"]
    .fillna(1)
)


missing_candidate = (
    gt_detection_3dfin["Tree_ID"].isna()
)


undersegmented = (
    gt_detection_3dfin["Tree_ID"].notna()
    &
    (gt_detection_3dfin["n_GT"] > 1)
)


# Begin with individual detection credit
# Then remove it for GT trees with no candidate or with an under-segmented shared candidate

gt_detection_3dfin["GT_detection_credit"] = 1

gt_detection_3dfin.loc[
    missing_candidate | undersegmented,
    "GT_detection_credit"
] = 0


gt_detection_3dfin["detection_status"] = (
    "detected_individual"
)


gt_detection_3dfin.loc[
    missing_candidate,
    "detection_status"
] = "not_detected_no_candidate"


gt_detection_3dfin.loc[
    undersegmented,
    "detection_status"
] = "undersegmented_not_individual"


# GT33 and GT34 are handled differently from an under-segmented pair
# Each has a separate 3DFin candidate for every scanner
# but the exact one-to-one identity of the two candidates was uncertain

ambiguous_3334 = (
    (gt_detection_3dfin["Terrain"] == "Challenging")
    &
    (gt_detection_3dfin["GT_ID"].isin([33, 34]))
)


require(
    (
        gt_detection_3dfin.loc[
            ambiguous_3334,
            "Tree_ID"
        ]
        .notna()
        .all()
    ),
    "GT33/GT34 should each have a 3DFin candidate.",
)


gt_detection_3dfin.loc[
    ambiguous_3334,
    "detection_status"
] = "detected_ambiguous_identity"


# Check that every GT × scanner record has a resolved detection decision

require(
    len(gt_detection_3dfin) == 174,
    "Expected 174 GT × scanner detection records.",
)

require(
    gt_detection_3dfin["GT_detection_credit"]
    .isin([0, 1])
    .all(),
    "Unresolved GT detection decisions were found.",
)


print("Challenging-terrain detection classifications:\n")

print(
    gt_detection_3dfin[
        gt_detection_3dfin["Terrain"] == "Challenging"
    ]
    .groupby(
        ["Scanner", "detection_status"]
    )
    .size()
)


print("\nFinal completeness:\n")


# Summarise the proportion of GT trees recovered as distinct individual 3DFin detections for each terrain and scanner

completeness_3dfin = (
    gt_detection_3dfin
    .groupby(
        ["Terrain", "Scanner"]
    )
    .agg(
        GT_trees=("GT_ID", "nunique"),
        Individually_detected=(
            "GT_detection_credit",
            "sum",
        ),
    )
    .reset_index()
)


completeness_3dfin["Completeness"] = (
    100
    * completeness_3dfin["Individually_detected"]
    / completeness_3dfin["GT_trees"]
)


print(
    completeness_3dfin.to_string(
        index=False,
        float_format=lambda x: f"{x:.1f}",
    )
)

Challenging-terrain detection classifications:

Scanner        detection_status             
Dual sensor    detected_ambiguous_identity       2
               detected_individual              19
               undersegmented_not_individual     6
Hovermap       detected_ambiguous_identity       2
               detected_individual              19
               undersegmented_not_individual     6
Single sensor  detected_ambiguous_identity       2
               detected_individual              19
               not_detected_no_candidate         2
               undersegmented_not_individual     4
dtype: int64

Final completeness:

      Terrain        Scanner  GT_trees  Individually_detected  Completeness
  Challenging    Dual sensor        27                     21          77.8
  Challenging       Hovermap        27                     21          77.8
  Challenging  Single sensor        27                     21          77.8
         Easy    Dual sensor         9                    

In [18]:
# Identify raw 3DFin candidates without a GT assignment
# A candidate is treated as GT-associated when its Tree ID appears anywhere in the final GT matching table
# This includes shared under-segmented candidates because they are still linked to known GT trees
# even though they do not count as individual detections

used_candidate_ids = (
    gt_matches_3dfin[
        gt_matches_3dfin["Tree_ID"].notna()
    ][
        [
            "Terrain",
            "Scanner",
            "Tree_ID",
        ]
    ]
    .drop_duplicates()
    .copy()
)


# Convert the Tree IDs to normal integers
used_candidate_ids["Tree_ID"] = (
    used_candidate_ids["Tree_ID"]
    .astype(int)
)

used_candidate_ids["used_in_GT_matching"] = True


# Mark each raw candidate according to whether it appears in the final GT matching table
candidate_audit = (
    raw_3dfin_candidates
    .merge(
        used_candidate_ids,
        on=[
            "Terrain",
            "Scanner",
            "Tree_ID",
        ],
        how="left",
    )
)


# Keep only raw candidates that were not assigned to any GT tree
unmatched_3dfin_candidates = (
    candidate_audit[
        candidate_audit["used_in_GT_matching"].isna()
    ]
    .copy()
    .sort_values(
        [
            "Terrain",
            "Scanner",
            "Tree_ID",
        ]
    )
    .reset_index(drop=True)
)


print(
    "Raw 3DFin candidates without a GT assignment:",
    len(unmatched_3dfin_candidates)
)


print("\nCounts by terrain and scanner:\n")

print(
    unmatched_3dfin_candidates
    .groupby(
        ["Terrain", "Scanner"]
    )
    .size()
)


print("\nCandidate details:\n")

print(
    unmatched_3dfin_candidates[
        [
            "Terrain",
            "Scanner",
            "Tree_ID",
            "TH",
            "DBH",
            "X",
            "Y",
        ]
    ]
    .to_string(index=False)
)

Raw 3DFin candidates without a GT assignment: 22

Counts by terrain and scanner:

Terrain       Scanner      
Challenging   Dual sensor      6
              Hovermap         6
              Single sensor    4
Intermediate  Dual sensor      2
              Hovermap         3
              Single sensor    1
dtype: int64

Candidate details:

      Terrain        Scanner  Tree_ID         TH       DBH          X         Y
  Challenging    Dual sensor        7  24.388353  0.078717  18.898779  3.731593
  Challenging    Dual sensor       10  25.064494  0.106752  19.651645  2.570457
  Challenging    Dual sensor       12  25.178746  0.069904  19.910139  1.713005
  Challenging    Dual sensor       13  24.654199  0.093350  20.204820  2.427762
  Challenging    Dual sensor       14   8.871810  0.000000  14.183858 -2.020458
  Challenging    Dual sensor       17  24.893231  0.076954  23.490222  1.870641
  Challenging       Hovermap        8  21.795499  0.083115  18.911322  3.711732
  Challenging     

In [19]:
# Build the final candidate-level 3DFin QC table
# Every original 3DFin candidate is assigned one final QC class:
# - validated_primary_match: a distinct candidate with an unambiguous GT match
# - detected_ambiguous_identity: one of the two distinct GT33/GT34 detections
# the latter includes cases where both trees were detected but their exact identities were uncertain
# - undersegmented_GT_trees: one candidate represents more than one GT tree
# - genuine_unrecorded_tree: a genuine tree detected by 3DFin but absent from the GT inventory
# No fragment or non-tree class is needed because the manual review did not identify these in 3DFin 

candidate_3dfin = raw_3dfin_candidates.copy()


# Add the valid GT-associated individual detections

valid_matches = (
    gt_detection_3dfin[
        (gt_detection_3dfin["GT_detection_credit"] == 1)
        &
        (gt_detection_3dfin["Tree_ID"].notna())
    ][
        [
            "Terrain",
            "Scanner",
            "Tree_ID",
            "GT_ID",
            "detection_status",
        ]
    ]
    .copy()
)


valid_matches["Tree_ID"] = (
    valid_matches["Tree_ID"]
    .astype(int)
)


candidate_3dfin = (
    candidate_3dfin
    .merge(
        valid_matches,
        on=[
            "Terrain",
            "Scanner",
            "Tree_ID",
        ],
        how="left",
    )
)


# Add the candidates that represent more than one GT tree

underseg_candidate_ids = (
    shared_tree_ids[
        [
            "Terrain",
            "Scanner",
            "Tree_ID",
            "represented_GT_IDs",
        ]
    ]
    .copy()
)


underseg_candidate_ids["Tree_ID"] = (
    underseg_candidate_ids["Tree_ID"]
    .astype(int)
)


underseg_candidate_ids[
    "is_undersegmented"
] = True


candidate_3dfin = (
    candidate_3dfin
    .merge(
        underseg_candidate_ids,
        on=[
            "Terrain",
            "Scanner",
            "Tree_ID",
        ],
        how="left",
    )
)


# Start unmatched candidates as genuine unrecorded trees
# Then replace this class where a GT match or under-segmentation is known

candidate_3dfin["QC_class"] = (
    "genuine_unrecorded_tree"
)


# Distinct candidates with a clear one-to-one GT match

normal_match = (
    candidate_3dfin["GT_ID"].notna()
    &
    (
        candidate_3dfin["detection_status"]
        == "detected_individual"
    )
)

candidate_3dfin.loc[
    normal_match,
    "QC_class"
] = "validated_primary_match"


# GT33 and GT34 have two distinct candidates, but their exact one-to-one identities could not be confirmed

ambiguous_match = (
    candidate_3dfin["GT_ID"].notna()
    &
    (
        candidate_3dfin["detection_status"]
        == "detected_ambiguous_identity"
    )
)

candidate_3dfin.loc[
    ambiguous_match,
    "QC_class"
] = "detected_ambiguous_identity"


# Shared candidates representing multiple GT trees are treated as under-segmentation errors

underseg_mask = (
    candidate_3dfin["is_undersegmented"]
    .fillna(False)
)

candidate_3dfin.loc[
    underseg_mask,
    "QC_class"
] = "undersegmented_GT_trees"


# Add an explanation of what each final class represents

CLASS_NOTES = {
    "validated_primary_match":
        "Distinct 3DFin candidate with an unambiguous GT match",

    "detected_ambiguous_identity":
        "Distinct GT33/GT34 tree instance; exact GT identity uncertain",

    "undersegmented_GT_trees":
        "Single 3DFin candidate represents multiple GT trees",

    "genuine_unrecorded_tree":
        "Manually reviewed genuine tree absent from GT inventory",
}


candidate_3dfin["QC_note"] = (
    candidate_3dfin["QC_class"]
    .map(CLASS_NOTES)
)


# Check that every original candidate has one resolved QC class

require(
    len(candidate_3dfin) == 186,
    "Expected exactly 186 candidate-level records.",
)

require(
    candidate_3dfin["QC_class"]
    .notna()
    .all(),
    "At least one 3DFin candidate has no final QC class.",
)


print(
    "Final 3DFin candidate classes:\n"
)

print(
    candidate_3dfin["QC_class"]
    .value_counts()
)


print(
    "\nClasses by terrain and scanner:\n"
)

print(
    candidate_3dfin
    .groupby(
        [
            "Terrain",
            "Scanner",
            "QC_class",
        ]
    )
    .size()
)

Final 3DFin candidate classes:

validated_primary_match        150
genuine_unrecorded_tree         22
undersegmented_GT_trees          8
detected_ambiguous_identity      6
Name: QC_class, dtype: int64

Classes by terrain and scanner:

Terrain       Scanner        QC_class                   
Challenging   Dual sensor    detected_ambiguous_identity     2
                             genuine_unrecorded_tree         6
                             undersegmented_GT_trees         3
                             validated_primary_match        19
              Hovermap       detected_ambiguous_identity     2
                             genuine_unrecorded_tree         6
                             undersegmented_GT_trees         3
                             validated_primary_match        19
              Single sensor  detected_ambiguous_identity     2
                             genuine_unrecorded_tree         4
                             undersegmented_GT_trees         2
               

In [20]:
# Calculate final 3DFin completeness and correctness
# Strict correctness measures the proportion of all 3DFin candidates that are valid individual detections of GT trees
# Adjusted correctness removes genuine unreferenced trees from the denominator because they are real detections
# Under-segmented candidates remain penalised

def summarise_3dfin_candidates(group):

    classes = group["QC_class"]

    # Count candidates that correspond to individually detected GT trees, including the GT33/GT34 ambiguous-identity cases
    matched = (
        classes.isin(
            [
                "validated_primary_match",
                "detected_ambiguous_identity",
            ]
        )
    ).sum()

    # Count genuine trees that were detected but were not present in the field-reference inventory
    genuine = (
        classes == "genuine_unrecorded_tree"
    ).sum()

    # Count under-segmented candidates, where one 3DFin output represents more than one GT tree
    invalid = (
        classes == "undersegmented_GT_trees"
    ).sum()

    return pd.Series({
        "Total_candidates": len(group),
        "Matched_candidates": matched,
        "Genuine_unreferenced": genuine,
        "Invalid_underseg_candidates": invalid,
    })


candidate_counts_3dfin = (
    candidate_3dfin
    .groupby(
        ["Terrain", "Scanner"]
    )
    .apply(summarise_3dfin_candidates)
    .reset_index()
)


# Combine candidate-level counts with the GT-level completeness results calculated above
detection_metrics_3dfin = (
    completeness_3dfin
    .merge(
        candidate_counts_3dfin,
        on=[
            "Terrain",
            "Scanner",
        ],
        how="left",
    )
)


# The number of valid matched candidates should equal the number of GT trees recovered as distinct individual detections
require(
    (
        detection_metrics_3dfin[
            "Matched_candidates"
        ]
        ==
        detection_metrics_3dfin[
            "Individually_detected"
        ]
    ).all(),
    (
        "Matched candidate counts do not agree with "
        "GT-level completeness."
    ),
)


# Check that every original candidate is represented by one of the final candidate classes
require(
    (
        detection_metrics_3dfin[
            "Matched_candidates"
        ]
        +
        detection_metrics_3dfin[
            "Genuine_unreferenced"
        ]
        +
        detection_metrics_3dfin[
            "Invalid_underseg_candidates"
        ]
        ==
        detection_metrics_3dfin[
            "Total_candidates"
        ]
    ).all(),
    "Candidate classifications do not sum to the raw totals.",
)


# Strict correctness keeps every 3DFin candidate in the denominator
detection_metrics_3dfin[
    "Strict_correctness"
] = (
    100
    * detection_metrics_3dfin[
        "Matched_candidates"
    ]
    / detection_metrics_3dfin[
        "Total_candidates"
    ]
)


# Adjusted correctness excludes genuine unreferenced trees from the denominator but continues to penalise under-segmentation
detection_metrics_3dfin[
    "Adjusted_correctness"
] = (
    100
    * detection_metrics_3dfin[
        "Matched_candidates"
    ]
    /
    (
        detection_metrics_3dfin[
            "Matched_candidates"
        ]
        +
        detection_metrics_3dfin[
            "Invalid_underseg_candidates"
        ]
    )
)


# Round the percentage metrics for the final summary table
for column in [
    "Completeness",
    "Strict_correctness",
    "Adjusted_correctness",
]:
    detection_metrics_3dfin[column] = (
        detection_metrics_3dfin[column]
        .round(1)
    )


print(
    detection_metrics_3dfin
    .to_string(index=False)
)

      Terrain        Scanner  GT_trees  Individually_detected  Completeness  Total_candidates  Matched_candidates  Genuine_unreferenced  Invalid_underseg_candidates  Strict_correctness  Adjusted_correctness
  Challenging    Dual sensor        27                     21          77.8                30                  21                     6                            3                70.0                  87.5
  Challenging       Hovermap        27                     21          77.8                30                  21                     6                            3                70.0                  87.5
  Challenging  Single sensor        27                     21          77.8                27                  21                     4                            2                77.8                  91.3
         Easy    Dual sensor         9                      9         100.0                 9                   9                     0                            0        

In [21]:
# Reconstruct the 3DFin measurement-analysis samples
# Measurement retention is handled separately from tree detection
# TH before QC requires:
# - HIGH-confidence tree identity
# - an available GT height
# - a valid automatic TH estimates from all three scanners
# DBH requires:
# HIGH-confidence tree identity
# an available GT DBH
# a valid non-zero DBH estimates from all three scanners
# The DBH sample is unchanged by the TH QC step because the manual 3DFin intervention targeted tree-height attribution

sample_rows = []


for terrain, source_df in manual_3dfin_clean.items():

    # Trees without a GT record cannot be used to assess measurement accuracy
    df = source_df[
        source_df["GT"].notna()
    ].copy()


    high_confidence = (
        df["CONF"] == "HIGH"
    )


    # Define the TH sample before manual attribution QC
    th_pre = (
        high_confidence
        &
        df["GT TH"].notna()
        &
        df["J TH"].notna()
        &
        df["D TH"].notna()
        &
        df["H TH"].notna()
        &
        (df["J TH"] > 0)
        &
        (df["D TH"] > 0)
        &
        (df["H TH"] > 0)
    )


    # Define the final TH sample after manual attribution QC
    th_post = (
        high_confidence
        &
        df["TH_validated"]
        &
        df["GT TH"].notna()
    )


    # The earlier post-QC workflow expected every retained tree to have a TH estimate from all three scanners
    require(
        (
            df.loc[
                th_post,
                ["J TH", "D TH", "H TH"]
            ]
            .notna()
            .all()
            .all()
        ),
        (
            f"{terrain}: a manually validated TH tree "
            "is missing a scanner TH value."
        ),
    )


    # Define the common DBH sample used throughout the analysis
    dbh = (
        high_confidence
        &
        df["GT DBH"].notna()
        &
        df["J DBH"].notna()
        &
        df["D DBH"].notna()
        &
        df["H DBH"].notna()
        &
        (df["J DBH"] > 0)
        &
        (df["D DBH"] > 0)
        &
        (df["H DBH"] > 0)
    )


    # Store one set of inclusion decisions for each GT tree
    for _, row in df.iterrows():

        sample_rows.append({
            "Terrain": terrain,
            "GT_ID": int(row["GT"]),
            "TH_pre_QC": bool(
                th_pre.loc[row.name]
            ),
            "TH_post_QC": bool(
                th_post.loc[row.name]
            ),
            "DBH": bool(
                dbh.loc[row.name]
            ),
        })


measurement_samples_3dfin = pd.DataFrame(
    sample_rows
)


print(
    "Measurement-sample reconstruction complete."
)

Measurement-sample reconstruction complete.


In [22]:
# Check the final sample sizes and GT trees retained for each measurement dataset before building the per-tree analysis tables

for terrain in [
    "Easy",
    "Intermediate",
    "Challenging",
]:

    df = measurement_samples_3dfin[
        measurement_samples_3dfin["Terrain"] == terrain
    ]

    print("\n" + "=" * 60)
    print(terrain.upper())
    print("=" * 60)

    for sample in [
        "TH_pre_QC",
        "TH_post_QC",
        "DBH",
    ]:

        retained = (
            df.loc[
                df[sample],
                "GT_ID"
            ]
            .astype(int)
            .tolist()
        )

        print(
            f"{sample:12s} "
            f"n = {len(retained):2d} | "
            f"GT = {retained}"
        )


EASY
TH_pre_QC    n =  9 | GT = [6, 13, 17, 20, 21, 27, 31, 33, 34]
TH_post_QC   n =  4 | GT = [6, 21, 27, 33]
DBH          n =  9 | GT = [6, 13, 17, 20, 21, 27, 31, 33, 34]

INTERMEDIATE
TH_pre_QC    n = 22 | GT = [1, 2, 4, 5, 8, 9, 11, 14, 15, 16, 17, 19, 20, 22, 23, 25, 26, 27, 28, 29, 30, 34]
TH_post_QC   n = 12 | GT = [1, 2, 9, 11, 17, 20, 22, 25, 26, 27, 30, 34]
DBH          n = 21 | GT = [1, 2, 4, 5, 8, 9, 11, 14, 15, 16, 17, 19, 20, 22, 23, 25, 27, 28, 29, 30, 34]

CHALLENGING
TH_pre_QC    n = 19 | GT = [1, 2, 3, 5, 6, 8, 9, 13, 14, 16, 18, 19, 20, 25, 26, 28, 29, 30, 37]
TH_post_QC   n = 13 | GT = [2, 3, 6, 8, 9, 14, 16, 18, 19, 20, 28, 29, 30]
DBH          n = 18 | GT = [1, 2, 3, 5, 6, 8, 9, 13, 14, 16, 18, 19, 20, 26, 28, 29, 30, 37]


In [23]:
# Inspect the earlier 3DFin validation outputs
# These files are used only to check that the reconstructed workflow reproduces the earlier results
# They are not used as inputs when deciding which observations are retained

validation_3dfin = {}


for name, path in VALIDATION_3DFIN_FILES.items():

    df = pd.read_csv(path)

    validation_3dfin[name] = df

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Rows:", len(df))
    print("Columns:", list(df.columns))

    # Show the main grouping values present in each validation file so its structure can be compared with the rebuilt data
    for column in [
        "Terrain",
        "Scanner",
        "Attribute",
        "Stage",
    ]:
        if column in df.columns:
            print(
                f"{column}:",
                sorted(
                    df[column]
                    .dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )
            )


pre_per_tree
Rows: 294
Columns: ['Terrain', 'Scanner', 'Attribute', 'Stage', 'GT', 'GT_value', 'Predicted_value', 'Error', 'ID_CONF']
Terrain: ['Challenging', 'Easy', 'Intermediate']
Scanner: ['Dvojskenner', 'Hovermap', 'Jednoskenner']
Attribute: ['DBH', 'TH']
Stage: ['Pre-TH-QC']

pre_summary
Rows: 18
Columns: ['Terrain', 'Scanner', 'Attribute', 'Stage', 'n', 'RMSE', 'MAE', 'Bias', 'R2']
Terrain: ['Challenging', 'Easy', 'Intermediate']
Scanner: ['Dvojskenner', 'Hovermap', 'Jednoskenner']
Attribute: ['DBH', 'TH']
Stage: ['Pre-TH-QC']

post_per_tree
Rows: 231
Columns: ['Terrain', 'Scanner', 'Attribute', 'GT', 'GT_value', 'Predicted_value', 'Error', 'ID_CONF', 'TH_validated']
Terrain: ['Challenging', 'Easy', 'Intermediate']
Scanner: ['Dvojskenner', 'Hovermap', 'Jednoskenner']
Attribute: ['DBH', 'TH']

post_summary
Rows: 18
Columns: ['Terrain', 'Scanner', 'Attribute', 'Stage', 'n', 'RMSE', 'MAE', 'Bias', 'R2']
Terrain: ['Challenging', 'Easy', 'Intermediate']
Scanner: ['Dvojskenner', 'Hov

In [24]:
# Check the per-tree sample sizes in the earlier validation outputs
# so they can be compared with the reconstructed measurement samples

for name in [
    "pre_per_tree",
    "post_per_tree",
]:

    df = validation_3dfin[name]

    print("\n" + name)

    print(
        df
        .groupby(
            [
                "Terrain",
                "Attribute",
                "Scanner",
            ]
        )
        .size()
    )


pre_per_tree
Terrain       Attribute  Scanner     
Challenging   DBH        Dvojskenner     18
                         Hovermap        18
                         Jednoskenner    18
              TH         Dvojskenner     19
                         Hovermap        19
                         Jednoskenner    19
Easy          DBH        Dvojskenner      9
                         Hovermap         9
                         Jednoskenner     9
              TH         Dvojskenner      9
                         Hovermap         9
                         Jednoskenner     9
Intermediate  DBH        Dvojskenner     21
                         Hovermap        21
                         Jednoskenner    21
              TH         Dvojskenner     22
                         Hovermap        22
                         Jednoskenner    22
dtype: int64

post_per_tree
Terrain       Attribute  Scanner     
Challenging   DBH        Dvojskenner     18
                         Hovermap        18
  

In [25]:
# Build the clean 3DFin per-tree measurement dataset
# Each row represents one GT tree, scanner, attribute and QC stage
# Error is calculated as scanner estimate minus field-reference value
# Positive values indicate overestimation and negative values indicate underestimation

SCANNER_MEASUREMENTS = {
    "Single sensor": {
        "tree_id": "J",
        "TH": "J TH",
        "DBH": "J DBH",
    },
    "Dual sensor": {
        "tree_id": "D",
        "TH": "D TH",
        "DBH": "D DBH",
    },
    "Hovermap": {
        "tree_id": "H",
        "TH": "H TH",
        "DBH": "H DBH",
    },
}


measurement_rows = []


for terrain, df in manual_3dfin_clean.items():

    gt_rows = df[
        df["GT"].notna()
    ].copy()

    for _, row in gt_rows.iterrows():

        gt_id = int(row["GT"])

        # Retrieve the previously defined sample-inclusion decisions for this GT tree
        sample_info = (
            measurement_samples_3dfin[
                (measurement_samples_3dfin["Terrain"] == terrain)
                &
                (measurement_samples_3dfin["GT_ID"] == gt_id)
            ]
            .iloc[0]
        )

        for scanner, columns in SCANNER_MEASUREMENTS.items():

            tree_id = clean_tree_id(
                row[columns["tree_id"]]
            )

            # Add TH measurements from the sample before manual tree-height attribution QC
            if sample_info["TH_pre_QC"]:

                gt_value = row["GT TH"]
                predicted = row[columns["TH"]]

                measurement_rows.append({
                    "Terrain": terrain,
                    "Scanner": scanner,
                    "Tree_ID": tree_id,
                    "GT_ID": gt_id,
                    "Attribute": "TH",
                    "Stage": "Pre-TH-QC",
                    "GT_value": gt_value,
                    "Predicted_value": predicted,
                    "Error": predicted - gt_value,
                    "Absolute_error": abs(
                        predicted - gt_value
                    ),
                    "ID_confidence": row["CONF"],
                    "TH_validated": row["TH_validated"],
                })


            # Add TH measurements retained after manual attribution QC
            if sample_info["TH_post_QC"]:

                gt_value = row["GT TH"]
                predicted = row[columns["TH"]]

                measurement_rows.append({
                    "Terrain": terrain,
                    "Scanner": scanner,
                    "Tree_ID": tree_id,
                    "GT_ID": gt_id,
                    "Attribute": "TH",
                    "Stage": "Post-QC",
                    "GT_value": gt_value,
                    "Predicted_value": predicted,
                    "Error": predicted - gt_value,
                    "Absolute_error": abs(
                        predicted - gt_value
                    ),
                    "ID_confidence": row["CONF"],
                    "TH_validated": True,
                })


            # The DBH sample is unchanged by the TH-attribution QC so the same measurements are stored at both stages
            if sample_info["DBH"]:

                gt_value = row["GT DBH"]
                predicted = row[columns["DBH"]]

                for stage in [
                    "Pre-TH-QC",
                    "Post-QC",
                ]:

                    measurement_rows.append({
                        "Terrain": terrain,
                        "Scanner": scanner,
                        "Tree_ID": tree_id,
                        "GT_ID": gt_id,
                        "Attribute": "DBH",
                        "Stage": stage,
                        "GT_value": gt_value,
                        "Predicted_value": predicted,
                        "Error": predicted - gt_value,
                        "Absolute_error": abs(
                            predicted - gt_value
                        ),
                        "ID_confidence": row["CONF"],
                        "TH_validated": np.nan,
                    })


measurements_3dfin = pd.DataFrame(
    measurement_rows
)


# Check that the reconstructed pre- and post-QC datasets contain the expected total number of measurement records
require(
    len(measurements_3dfin) == 525,
    (
        "Expected 525 records across the pre- and "
        "post-QC 3DFin datasets."
    ),
)


print(
    "Clean 3DFin measurement records:",
    len(measurements_3dfin)
)

print()

print(
    measurements_3dfin
    .groupby(
        [
            "Stage",
            "Attribute",
        ]
    )
    .size()
)

Clean 3DFin measurement records: 525

Stage      Attribute
Post-QC    DBH          144
           TH            87
Pre-TH-QC  DBH          144
           TH           150
dtype: int64


In [26]:
# Validate the reconstructed per-tree values against the earlier 3DFin outputs
# The earlier scripts used different scanner names
# The current descriptive names are mapped back only for this validation step

LEGACY_SCANNER_NAMES = {
    "Single sensor": "Jednoskenner",
    "Dual sensor": "Dvojskenner",
    "Hovermap": "Hovermap",
}


def prepare_for_validation(df, stage):

    out = df[
        df["Stage"] == stage
    ].copy()

    # Match the scanner labels used in the earlier output files
    out["Scanner"] = (
        out["Scanner"]
        .map(LEGACY_SCANNER_NAMES)
    )

    out = out.rename(
        columns={
            "GT_ID": "GT",
            "ID_confidence": "ID_CONF",
        }
    )

    # Standardise GT IDs and confidence labels before comparison
    out["GT"] = (
        pd.to_numeric(
            out["GT"],
            errors="coerce"
        )
        .astype(int)
    )

    out["ID_CONF"] = (
        out["ID_CONF"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    return out


def compare_per_tree(
    reconstructed,
    historical,
    label,
):

    historical = historical.copy()

    # Standardise the earlier output to the same format as the reconstructed dataset
    historical["GT"] = (
        pd.to_numeric(
            historical["GT"],
            errors="coerce"
        )
        .astype(int)
    )

    historical["ID_CONF"] = (
        historical["ID_CONF"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    keys = [
        "Terrain",
        "Scanner",
        "Attribute",
        "GT",
    ]

    compare_columns = [
        "GT_value",
        "Predicted_value",
        "Error",
    ]

    # Match records by terrain, scanner, attribute and GT tree
    #so that both the retained sample and the measurement values can be checked
    merged = reconstructed[
        keys
        +
        compare_columns
        +
        ["ID_CONF"]
    ].merge(
        historical[
            keys
            +
            compare_columns
            +
            ["ID_CONF"]
        ],
        on=keys,
        how="outer",
        suffixes=(
            "_new",
            "_historical",
        ),
        indicator=True,
    )


    # Every reconstructed record should have an exact counterpart in the earlier output, and vice versa
    require(
        (merged["_merge"] == "both").all(),
        (
            f"{label}: reconstructed and earlier "
            "records do not contain the same trees."
        ),
    )


    # Check that the GT values, scanner estimates and resulting errors reproduce the earlier per-tree values
    for column in compare_columns:

        difference = np.abs(
            merged[f"{column}_new"]
            -
            merged[f"{column}_historical"]
        )

        require(
            np.all(
                difference < 1e-10
            ),
            (
                f"{label}: values differ in "
                f"'{column}'."
            ),
        )


    # Tree-identity confidence should also match the earlier output
    require(
        (
            merged["ID_CONF_new"]
            ==
            merged["ID_CONF_historical"]
        ).all(),
        (
            f"{label}: confidence labels differ."
        ),
    )


    print(
        f"PASS: {label} matches earlier "
        f"per-tree output ({len(merged)} rows)."
    )


new_pre = prepare_for_validation(
    measurements_3dfin,
    "Pre-TH-QC",
)

new_post = prepare_for_validation(
    measurements_3dfin,
    "Post-QC",
)


compare_per_tree(
    new_pre,
    validation_3dfin["pre_per_tree"],
    "Pre-TH-QC",
)

compare_per_tree(
    new_post,
    validation_3dfin["post_per_tree"],
    "Post-QC",
)

PASS: Pre-TH-QC matches earlier per-tree output (294 rows).
PASS: Post-QC matches earlier per-tree output (231 rows).


In [27]:
# Calculate the 3DFin summary accuracy statistics
# The same metrics used in the earlier analysis are calculated: RMSE, MAE, Bias and R²
# Error is defined as scanner estimate minus field-reference value

def calculate_accuracy_metrics(group):

    gt = group["GT_value"].to_numpy(dtype=float)
    predicted = group["Predicted_value"].to_numpy(dtype=float)

    error = predicted - gt

    # Root mean square error
    rmse = np.sqrt(
        np.mean(error ** 2)
    )

    # Mean absolute error
    mae = np.mean(
        np.abs(error)
    )

    # Mean signed error. Positive values indicate overestimation and negative values indicate underestimation
    bias = np.mean(error)

    # Prediction-based R² calculated as 1 - SSE / SST
    ss_res = np.sum(
        (gt - predicted) ** 2
    )

    ss_tot = np.sum(
        (gt - np.mean(gt)) ** 2
    )

    if ss_tot == 0:
        r2 = np.nan
    else:
        r2 = 1 - (ss_res / ss_tot)

    return pd.Series({
        "n": len(group),
        "RMSE": rmse,
        "MAE": mae,
        "Bias": bias,
        "R2": r2,
    })


# Calculate one set of accuracy metrics for every terrain, scanner, attribute and QC-stage combination
summary_3dfin = (
    measurements_3dfin
    .groupby(
        [
            "Terrain",
            "Scanner",
            "Attribute",
            "Stage",
        ]
    )
    .apply(calculate_accuracy_metrics)
    .reset_index()
)


# Three terrains × three scanners × two attributes × two stages should give 36 summary rows
require(
    len(summary_3dfin) == 36,
    "Expected 36 scanner × terrain × attribute × stage summaries.",
)


print(
    "3DFin summary rows:",
    len(summary_3dfin)
)

3DFin summary rows: 36


In [28]:
# Validate the reconstructed summary statistics against the earlier 3DFin outputs

historical_summary = pd.concat(
    [
        validation_3dfin["pre_summary"],
        validation_3dfin["post_summary"],
    ],
    ignore_index=True,
)


new_summary_validation = (
    summary_3dfin.copy()
)


# Match the scanner names used in the earlier summary files
new_summary_validation["Scanner"] = (
    new_summary_validation["Scanner"]
    .map(LEGACY_SCANNER_NAMES)
)


summary_keys = [
    "Terrain",
    "Scanner",
    "Attribute",
    "Stage",
]


# Match each reconstructed summary row to the corresponding row in the earlier outputs
summary_comparison = (
    new_summary_validation
    .merge(
        historical_summary,
        on=summary_keys,
        how="outer",
        suffixes=(
            "_new",
            "_historical",
        ),
        indicator=True,
    )
)


# The reconstructed and earlier summaries should contain the same terrain-scanner-attribute-stage combinations
require(
    (summary_comparison["_merge"] == "both").all(),
    (
        "New and earlier summary files do not "
        "contain the same groups."
    ),
)


# Check that the retained sample sizes are identical
require(
    (
        summary_comparison["n_new"]
        ==
        summary_comparison["n_historical"]
    ).all(),
    "Earlier and reconstructed sample sizes differ.",
)


# Check that all calculated accuracy metrics reproduce the earlier values
for metric in [
    "RMSE",
    "MAE",
    "Bias",
    "R2",
]:

    difference = np.abs(
        summary_comparison[f"{metric}_new"]
        -
        summary_comparison[f"{metric}_historical"]
    )

    require(
        np.all(
            difference.fillna(0) < 1e-10
        ),
        (
            f"Earlier and reconstructed "
            f"{metric} values differ."
        ),
    )


print(
    "PASS: all 36 reconstructed summary rows match "
    "the earlier 3DFin results."
)

PASS: all 36 reconstructed summary rows match the earlier 3DFin results.


In [29]:
# Prepare the final processed 3DFin outputs
# These tables are built entirely from the original 3DFin spreadsheets and the manually checked terrain workbooks
# The earlier validation CSVs are used only for comparison and do not contribute to the final processed datasets
# Keep the per-tree measurement fields needed for the later analysis
measurements_3dfin_final = (
    measurements_3dfin[
        [
            "Terrain",
            "Scanner",
            "GT_ID",
            "Tree_ID",
            "Attribute",
            "Stage",
            "GT_value",
            "Predicted_value",
            "Error",
            "Absolute_error",
            "ID_confidence",
            "TH_validated",
        ]
    ]
    .copy()
)


# Keep the sample-membership decisions showing which GT trees were retained for TH before QC, TH after QC and DBH
measurement_samples_3dfin_final = (
    measurement_samples_3dfin[
        [
            "Terrain",
            "GT_ID",
            "TH_pre_QC",
            "TH_post_QC",
            "DBH",
        ]
    ]
    .copy()
)


# Prepare the candidate-level detection table
candidate_3dfin_final = candidate_3dfin.copy()


# Add boolean fields that make the final interpretation of each candidate easier to use in later analysis

candidate_3dfin_final[
    "valid_individual_detection"
] = (
    candidate_3dfin_final["QC_class"]
    .isin(
        [
            "validated_primary_match",
            "detected_ambiguous_identity",
        ]
    )
)


candidate_3dfin_final[
    "genuine_unreferenced"
] = (
    candidate_3dfin_final["QC_class"]
    ==
    "genuine_unrecorded_tree"
)


candidate_3dfin_final[
    "invalid_detection"
] = (
    candidate_3dfin_final["QC_class"]
    ==
    "undersegmented_GT_trees"
)


# Keep the final completeness and correctness metrics required for each terrain-scanner combination
detection_metrics_3dfin_final = (
    detection_metrics_3dfin[
        [
            "Terrain",
            "Scanner",
            "GT_trees",
            "Individually_detected",
            "Total_candidates",
            "Matched_candidates",
            "Genuine_unreferenced",
            "Invalid_underseg_candidates",
            "Completeness",
            "Strict_correctness",
            "Adjusted_correctness",
        ]
    ]
    .copy()
)


print("Final 3DFin processed tables prepared.")

Final 3DFin processed tables prepared.


In [30]:
# Save the final processed 3DFin datasets

OUTPUT_FILES = {
    "3dfin_measurements_per_tree.csv":
        measurements_3dfin_final,

    "3dfin_measurement_samples.csv":
        measurement_samples_3dfin_final,

    "3dfin_candidate_qc_master.csv":
        candidate_3dfin_final,

    "3dfin_detection_metrics.csv":
        detection_metrics_3dfin_final,
}


for filename, dataframe in OUTPUT_FILES.items():

    output_path = (
        PROCESSED_3DFIN_DIR
        / filename
    )

    dataframe.to_csv(
        output_path,
        index=False,
    )

    print(
        f"Saved {filename:35s} "
        f"({len(dataframe)} rows)"
    )

Saved 3dfin_measurements_per_tree.csv     (525 rows)
Saved 3dfin_measurement_samples.csv       (58 rows)
Saved 3dfin_candidate_qc_master.csv       (186 rows)
Saved 3dfin_detection_metrics.csv         (9 rows)


In [31]:
# Final output verification
# Reload each processed file and check that it was created successfully with the expected number of rows

EXPECTED_OUTPUTS = {
    "3dfin_measurements_per_tree.csv": 525,
    "3dfin_measurement_samples.csv": 58,
    "3dfin_candidate_qc_master.csv": 186,
    "3dfin_detection_metrics.csv": 9,
}


for filename, expected_rows in EXPECTED_OUTPUTS.items():

    path = (
        PROCESSED_3DFIN_DIR
        / filename
    )

    require(
        path.exists(),
        f"Processed output was not created: {filename}",
    )

    check = pd.read_csv(path)

    require(
        len(check) == expected_rows,
        (
            f"{filename}: expected "
            f"{expected_rows} rows but found {len(check)}."
        ),
    )

    print(f"OK {filename}")


print("\n3DFin data preparation complete.")

OK 3dfin_measurements_per_tree.csv
OK 3dfin_measurement_samples.csv
OK 3dfin_candidate_qc_master.csv
OK 3dfin_detection_metrics.csv

3DFin data preparation complete.
